# BERT for Quora Question Pairs

In [1]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from datasets import load_dataset
import lightning as L

In [2]:
DATASET_NAME = "AlekseyKorshuk/quora-question-pairs"
MODEL_NAME = "google-bert/bert-base-uncased"
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 4
D_MODEL = 768
LEARNING_RATE = 1e-3
MAX_EPOCHS = 3

## 1. Prepare dataset

In [3]:
ds = load_dataset(DATASET_NAME, split="train")

In [4]:
ds

Dataset({
    features: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'],
    num_rows: 404290
})

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
ds_cleaned = ds.filter(
    lambda row: isinstance(row["question1"], str) and isinstance(row["question2"], str)
)
ds_tokenized = ds_cleaned.map(
    lambda row: tokenizer(row["question1"], row["question2"]),
    batched=True,
    remove_columns=["question1", "question2"],
)

In [7]:
ds_split_1 = ds_tokenized.train_test_split(0.3)
ds_split_2 = ds_split_1["test"].train_test_split(0.5)
ds_train, ds_val, ds_test = ds_split_1["train"], ds_split_2["train"], ds_split_2["test"]

In [8]:
ds_train, ds_val, ds_test

(Dataset({
     features: ['id', 'qid1', 'qid2', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 283000
 }),
 Dataset({
     features: ['id', 'qid1', 'qid2', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 60643
 }),
 Dataset({
     features: ['id', 'qid1', 'qid2', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 60644
 }))

In [9]:
data_collator = DataCollatorWithPadding(tokenizer)

In [10]:
train_loader = DataLoader(
    ds_train,
    BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    ds_val,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    ds_test,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)

In [11]:
for batch in test_loader:
    yo = batch
    break

In [12]:
yo

{'id': tensor([ 44182, 327637, 174020, 234690,  45600,  46925, 367683,  78252, 239301,
        392844, 295037, 132335, 355029, 205283,  95775, 118743,   3114, 391303,
        288858, 394495,  65896,  39556, 273712,   4264, 127449, 113574, 250205,
         26173, 297621, 384548,   9379, 278884]), 'qid1': tensor([ 79347, 454083, 268302, 345255,  10674,   4752,  20620, 133462,  74646,
        525573,  62253, 211923, 484226, 308385,  40928,  54592,   6174, 368190,
        409873, 527384,  52237,   9299, 392218,   8431, 205170, 185577, 364017,
         48747, 420008,  92739,  18225, 398215]), 'qid2': tensor([ 79348, 454084, 268303, 345256,   8534,  83851, 304495, 133463,  73540,
        525574, 417024, 211924, 484227, 308386, 159652, 192934,   6175, 494051,
        409874, 359129,  86916,  71689, 392219,   8432, 205171, 185578,  70758,
         48748, 420009,  56363,  18226, 398216]), 'is_duplicate': tensor([0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1,
        0, 

## 2. Prepare model

In [19]:
class BertQuoraModel(L.LightningModule):
    def __init__(self, bert, d_model = D_MODEL):
        super().__init__()
        self.bert = bert
        self.d_model = d_model
        self.classifier = nn.Linear(d_model, 1)
    
    def forward(self, input_ids, attention_mask, token_type_ids):
        bert_output = self.bert(input_ids, attention_mask, token_type_ids)
        cls_token = bert_output.pooler_output
        logits = self.classifier(cls_token).reshape(-1)
        return logits
    
    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids, attention_mask, token_type_ids)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        
        preds = logits > 0
        accuracy = (preds == labels).float().mean()
        tp = preds.logical_and(labels).sum()
        fp = preds.logical_and(labels.logical_not()).sum()
        fn = preds.logical_not().logical_and(labels).sum()
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        f1 = 2 * precision * recall / (precision + recall)

        self.log("train_loss", loss)
        self.log("train_accuracy", accuracy)
        self.log("train_precision", precision)
        self.log("train_recall", recall)
        self.log("train_f1", f1)

        return loss
        
    
    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), LEARNING_RATE)
        return optimizer

In [20]:
bert = AutoModel.from_pretrained(MODEL_NAME)
bert.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [21]:
bert

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

## 3. Train model

In [22]:
model = BertQuoraModel(bert)

In [23]:
trainer = L.Trainer(max_epochs=1, limit_train_batches=100)
trainer.fit(model, train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert       │ BertModel │  109 M │ train │     0 │
│ 1 │ classifier │ Linear    │    769 │ train │     0 │
└───┴────────────┴───────────┴────────┴───────┴───────┘

Trainable params: 109 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 109 M                                                                                                
Total estimated model params size (MB): 437.932                                                                    
Modules in train mode: 229                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=1` reached.


In [24]:
yo["is_duplicate"].dtype

torch.int64